# Análise de Cotas de EEB, LR e Escoamento das Bacias

Este script processa arquivos geográficos para compilar dados das bacias hidrográficas e alternativas de linhas de recalque.

Entrada esperada:
- bacias
- coluna_bacias: nome da coluna do shape de bacias onde tem o nome das bacias
- eeb
- MDE.tif
- caminho_lr: linha de recalque

Saída (definido pelo usuário):
- Para cada bacia: cota da EEB, maior cota ao longo de cada linha de recalque, extensão da linha, e bacia de destino da linha.

O script associa automaticamente os dados às bacias e usa o MDE (com fallback online) para calcular elevações.

In [3]:
# ======================== COTAS — BLOCO LIMPO (copy/paste) ========================
# Imports únicos
import os
import copy
import re
import numpy as np
import pandas as pd
import requests
import rasterio
from rasterio.transform import rowcol
import pyproj
import geopandas as gpd
from shapely.geometry import Point, LineString

## Funções Auxiliares

In [8]:
# -----------------------------------------
# Helper simples de log
# -----------------------------------------
def _log(msg: str):
    print(f"[cotas] {msg}")

# -----------------------------------------
# CRS helpers (normaliza LOCAL_CS SIRGAS/UTM -> EPSG)
# -----------------------------------------
def _normalize_crs(obj_crs, fallback_epsg=None):
    try:
        if obj_crs:
            return pyproj.CRS.from_user_input(obj_crs)
    except Exception:
        pass
    if fallback_epsg is not None:
        try:
            return pyproj.CRS.from_epsg(fallback_epsg)
        except Exception:
            pass
    return None

def _guess_epsg_from_local_sirgas(crs_like):
    try:
        crs = pyproj.CRS.from_user_input(crs_like)
        wkt = crs.to_wkt()
        name = crs.name or ""
        blob = f"{name} {wkt}".upper()
    except Exception:
        return None
    if "SIRGAS" in blob and "UTM" in blob and "ZONE" in blob:
        m = re.search(r"ZONE\s+(\d+)", blob)
        if m:
            zone = int(m.group(1))
            epsg = 31960 + zone  # 21->31981, 22->31982, 23->31983...
            try:
                return pyproj.CRS.from_epsg(epsg)
            except Exception:
                return None
    return None

def get_raster_crs(mde_path, fallback_epsg=None):
    with rasterio.open(mde_path) as src:
        raw = src.crs
        norm = _normalize_crs(raw, fallback_epsg=None)
        if norm is not None and norm.to_epsg() is None:
            guess = _guess_epsg_from_local_sirgas(norm)
            if guess is not None:
                _log(f"CRS do MDT está em LOCAL_CS; normalizado para {guess.to_string()} (via heurística SIRGAS/UTM).")
                return guess
        if norm is None and fallback_epsg is not None:
            try:
                fb = pyproj.CRS.from_epsg(fallback_epsg)
                _log(f"ATENÇÃO: não consegui normalizar o CRS do MDT; usando fallback {fb.to_string()}.")
                return fb
            except Exception:
                pass
        if norm is None:
            _log("ATENÇÃO: não consegui normalizar o CRS do MDT (nem fallback).")
        else:
            _log(f"CRS do MDT detectado: {norm.to_string()}")
        return norm

def ensure_to_crs(gdf: gpd.GeoDataFrame, target_crs: pyproj.CRS, assume_if_none=True):
    if target_crs is None:
        _log("ERRO: target_crs=None em ensure_to_crs(). Retornando cópia sem alterar CRS.")
        return copy.deepcopy(gdf)
    target_norm = target_crs
    if target_crs.to_epsg() is None:
        guess = _guess_epsg_from_local_sirgas(target_crs)
        if guess is not None:
            _log(f"Normalizando CRS-alvo LOCAL_CS para {guess.to_string()} (EPSG) antes de reprojetar.")
            target_norm = guess
    if gdf.crs is None:
        if assume_if_none:
            _log(f"GeoDataFrame sem CRS. ASSUMINDO {target_norm.to_string()} (set_crs, sem reprojetar).")
            try:
                return gdf.set_crs(target_norm, allow_override=True)
            except Exception:
                _log("Falha ao set_crs(). Retornando cópia original.")
                return copy.deepcopy(gdf)
        else:
            _log("GeoDataFrame sem CRS e assume_if_none=False. Retornando cópia original.")
            return copy.deepcopy(gdf)
    try:
        src_crs = pyproj.CRS.from_user_input(gdf.crs)
        if src_crs == target_norm:
            return copy.deepcopy(gdf)
    except Exception:
        _log("Não consegui comparar CRS; tentando reprojetar assim mesmo.")
    try:
        _log(f"Reprojetando GDF de {gdf.crs} -> {target_norm.to_string()} ...")
        return gdf.to_crs(target_norm)
    except Exception as e:
        _log(f"Falha ao reprojetar ({e}); retornando cópia original.")
        return copy.deepcopy(gdf)

def align_gdf_to_raster_crs_or_fallback(gdf, mde_path, fallback_epsg=None, assume_if_none=True):
    out_crs = get_raster_crs(mde_path, fallback_epsg=fallback_epsg)
    if out_crs is None:
        _log("Não foi possível determinar CRS de saída. Retornando sem alterar.")
        return copy.deepcopy(gdf), None
    if out_crs.to_epsg() is None:
        guess = _guess_epsg_from_local_sirgas(out_crs)
        if guess is not None:
            out_crs = guess
    gdf2 = ensure_to_crs(gdf, out_crs, assume_if_none=assume_if_none)
    return gdf2, out_crs

# -----------------------------------------
# OpenTopodata fallback — simples + teimoso + cache
# -----------------------------------------
_session = requests.Session()
_session.headers.update({"User-Agent": "NEP-Cotas/1.0 (+contato@novaengevix.com.br)"})
_fb_cache: dict[tuple[float, float], float] = {}  # (lon_r5, lat_r5) -> elevation

def buscar_cota_online(lon, lat, timeout: float = 6.0) -> float:
    """
    Consulta SRTM90 (OpenTopodata) com 2 tentativas (exata + jitter) e cache por chave arredondada (5 casas).
    Retorna float (m) ou np.nan.
    """
    lon = float(lon); lat = float(lat)
    key = (round(lon, 5), round(lat, 5))
    if key in _fb_cache:
        return _fb_cache[key]
    def _req(lon_q: float, lat_q: float) -> float:
        url = f"https://api.opentopodata.org/v1/srtm90m?locations={lat_q},{lon_q}"
        try:
            r = _session.get(url, timeout=timeout)
            if r.status_code == 200:
                res = r.json().get("results")
                if res and (res[0].get("elevation") is not None):
                    return float(res[0]["elevation"])
        except Exception:
            pass
        return np.nan
    v = _req(lon, lat)
    if np.isnan(v):
        v = _req(lon + 1e-6, lat + 1e-6)
    _fb_cache[key] = v
    return v

# -----------------------------------------
# Interpolação por metros (precisa)
# -----------------------------------------
def interpolar_linha_por_metros(geom, intervalo_m=10.0):
    if geom is None or geom.is_empty:
        return []
    if geom.geom_type == "MultiLineString":
        coords = []
        for line in geom.geoms:
            coords.extend(line.coords)
        geom = LineString(coords)
    if geom.geom_type != "LineString":
        return []
    L = geom.length
    if L == 0:
        return [Point(geom.coords[0])]
    dists = np.arange(0.0, L, float(intervalo_m))
    pontos = [geom.interpolate(float(d), normalized=False) for d in dists]
    if pontos and pontos[-1].distance(Point(geom.coords[-1])) > 1e-9:
        pontos.append(Point(geom.coords[-1]))
    return pontos

# -----------------------------------------
# Atribuir bacia via sjoin (rápido)
# -----------------------------------------
def carregar_shapefile_com_bacia(caminho, bacias: gpd.GeoDataFrame, nome_col_bacia: str):
    _log(f"Lendo shapefile: {caminho}")
    gdf = gpd.read_file(caminho)
    if bacias.crs is None:
        raise ValueError("GeoDataFrame 'bacias' está sem CRS.")
    if gdf.crs != bacias.crs:
        _log(f"Reprojetando dado lido de {gdf.crs} -> {bacias.crs}")
        gdf = gdf.to_crs(bacias.crs)
    def ref_point(geom):
        if geom is None or geom.is_empty:
            return None
        gt = geom.geom_type
        if gt == "Point": return geom
        if gt == "MultiPoint": return list(geom.geoms)[0]
        if gt == "LineString": return Point(geom.coords[0])
        if gt == "MultiLineString":
            first = list(geom.geoms)[0]
            return Point(first.coords[0])
        return geom.representative_point()
    gdf_ref = gdf.copy()
    gdf_ref["__ponto_ref__"] = gdf_ref["geometry"].apply(ref_point)
    gdf_ref = gdf_ref.set_geometry("__ponto_ref__")
    if gdf_ref.crs != bacias.crs:
        gdf_ref = gdf_ref.set_crs(bacias.crs, allow_override=True)
    right = bacias[[nome_col_bacia, "geometry"]].copy().set_geometry("geometry")
    joined = gpd.sjoin(gdf_ref[["__ponto_ref__"]], right, how="left", predicate="within")
    gdf["nome_bacia"] = pd.Series(index=gdf.index, dtype="object")
    gdf.loc[joined.index, "nome_bacia"] = joined[nome_col_bacia].astype("object").values
    _log(f"sjoin concluído: {gdf['nome_bacia'].notna().sum()} feições com bacia.")
    return gdf

    
# --- DIAGNÓSTICO DE EXTENTS / CRS ---
with rasterio.open(mde_path) as _src:
    rb = _src.bounds
    print("[debug] Raster CRS:", _src.crs)
    print("[debug] Raster bounds (m):", (rb.left, rb.bottom, rb.right, rb.top))

print("[debug] EEB CRS lido:", eeb.crs)
print("[debug] EEB bounds (raw):", eeb.total_bounds)  # (minx, miny, maxx, maxy)

# Heurística p/ detectar graus vs metros
minx, miny, maxx, maxy = eeb.total_bounds
if all([-180 <= v <= 180 for v in [minx, maxx]]) and all([-90 <= v <= 90 for v in [miny, maxy]]):
    print("[debug] As EEBs parecem estar em GRAUS (lon/lat).")
else:
    print("[debug] As EEBs parecem estar em METROS (UTM).")

# -----------------------------------------
# Cota para pontos (EEB) — MDT + fallback em TODOS os zeros
# -----------------------------------------
def adicionar_cota(
    gdf_points: gpd.GeoDataFrame,
    mde_path,
    intervalo_debug=False,
    fallback_epsg=None
):
    _log("=== Iniciando adicionar_cota() ===")

    # Alinha as EEBs ao CRS do raster
    gdf_src, raster_crs = align_gdf_to_raster_crs_or_fallback(
        gdf_points, mde_path, fallback_epsg=fallback_epsg, assume_if_none=False
    )

    with rasterio.open(mde_path) as src:
        xs = np.asarray(gdf_src.geometry.x, dtype="float64")
        ys = np.asarray(gdf_src.geometry.y, dtype="float64")

        b = src.bounds
        mask_in = (xs >= b.left) & (xs <= b.right) & (ys >= b.bottom) & (ys <= b.top)

        mask_img = src.dataset_mask()
        band1_nodata = src.nodata

        cotas = np.full(xs.shape, np.nan, dtype="float64")
        status = np.full(xs.shape, "sem_dados", dtype=object)
        zeros_para_fallback_idx = []

        # --- MDT (amostragem) ---
        if mask_in.any():
            _log(f"Pontos dentro do raster para amostragem MDT: {int(mask_in.sum())}")

            rr, cc = rowcol(src.transform, xs[mask_in], ys[mask_in], op=np.floor)
            rr = np.clip(rr.astype(np.intp), 0, mask_img.shape[0] - 1)
            cc = np.clip(cc.astype(np.intp), 0, mask_img.shape[1] - 1)

            pix_ok = (mask_img[rr, cc] != 0)
            idx_in = np.where(mask_in)[0]

            # pixels válidos → sample
            if np.any(pix_ok):
                vals_ok = np.array(
                    [v[0] for v in src.sample(zip(xs[mask_in][pix_ok], ys[mask_in][pix_ok]))],
                    dtype="float64"
                )
                if band1_nodata is not None:
                    vals_ok = np.where(vals_ok == band1_nodata, 0.0, vals_ok)

                idx_global_ok = idx_in[pix_ok]
                cotas[idx_global_ok] = vals_ok
                status[idx_global_ok] = "mdt"

                zero_mask_ok = (vals_ok == 0.0)
                if np.any(zero_mask_ok):
                    zeros_para_fallback_idx.extend(idx_global_ok[zero_mask_ok].tolist())

            # pixels fora da máscara → 0.0 e vão ao fallback
            if np.any(~pix_ok):
                idx_global_nd = idx_in[~pix_ok]
                cotas[idx_global_nd] = 0.0
                status[idx_global_nd] = "mdt"
                zeros_para_fallback_idx.extend(idx_global_nd.tolist())

            _log(f"MDT aplicado em {int(mask_in.sum())} pontos (zeros a validar: {len(zeros_para_fallback_idx)}).")

        # --- Quem vai para o fallback ---
        fora_raster_idx = np.where(np.isnan(cotas))[0].tolist()
        zeros_para_fallback_idx = list(dict.fromkeys(zeros_para_fallback_idx))
        need_fb_idx = sorted(set(fora_raster_idx) | set(zeros_para_fallback_idx))
        _log(f"Pontos que irão ao fallback (fora raster + zeros): {len(need_fb_idx)}")

        if need_fb_idx:
            need_fb_mask = np.zeros(xs.shape[0], dtype=bool)
            need_fb_mask[need_fb_idx] = True

            lon = lat = None

            # (A) raster_crs -> WGS84
            try:
                base_crs = _normalize_crs(raster_crs)
                if base_crs is not None:
                    t_raster = pyproj.Transformer.from_crs(base_crs, "EPSG:4326", always_xy=True)
                    lon, lat = t_raster.transform(xs[need_fb_mask], ys[need_fb_mask])
                    _log("Transformação raster->WGS84 OK (primeira tentativa).")
            except Exception:
                lon = lat = None

            # (B) CRS original -> WGS84
            if (lon is None) or (lat is None):
                try:
                    crs_from_norm = _normalize_crs(gdf_points.crs)
                    if crs_from_norm is not None:
                        t_orig = pyproj.Transformer.from_crs(crs_from_norm, "EPSG:4326", always_xy=True)
                        lon, lat = t_orig.transform(
                            np.asarray(gdf_points.geometry.x)[need_fb_mask],
                            np.asarray(gdf_points.geometry.y)[need_fb_mask]
                        )
                        _log("Transformação original->WGS84 OK (segunda tentativa).")
                except Exception:
                    lon = lat = None

            # (C) heurística final
            if (lon is None) or (lat is None):
                cand_x = np.asarray(gdf_points.geometry.x)[need_fb_mask]
                cand_y = np.asarray(gdf_points.geometry.y)[need_fb_mask]
                if np.all((cand_x >= -180) & (cand_x <= 180)) and np.all((cand_y >= -90) & (cand_y <= 90)):
                    lon, lat = cand_x, cand_y
                    _log("Heurística: coordenadas já parecem lon/lat.")

            # === Fallback com jitter ===
            if (lon is not None) and (lat is not None):
                offsets = [(0.0, 0.0), (1e-6, 0.0), (0.0, 1e-6), (-1e-6, 0.0), (0.0, -1e-6)]
                fb_vals = np.full(len(lon), np.nan, dtype="float64")
                for dx, dy in offsets:
                    miss = np.isnan(fb_vals)
                    if not np.any(miss):
                        break
                    tries = np.array(
                        [buscar_cota_online(lx + dx, ly + dy) for lx, ly in zip(lon[miss], lat[miss])],
                        dtype="float64"
                    )
                    fb_vals[miss] = tries

                ok = ~np.isnan(fb_vals)
                idx = np.where(need_fb_mask)[0]
                if np.any(ok):
                    sel_idx = idx[ok]
                    cotas[sel_idx] = fb_vals[ok]
                    status[sel_idx] = "fallback"

                _log(f"Fallback aplicado em {int(ok.sum())} pontos; sem retorno em {int((~ok).sum())}.")

        out = copy.deepcopy(gdf_points)
        out["cota"] = cotas
        out["status_eeb"] = status
        _log("adicionar_cota() concluído.")
        return out



# -----------------------------------------
# Maior cota ao longo da LR — MDT + fallback em TODOS os zeros
# -----------------------------------------
def maior_cota_lr(lrs: gpd.GeoDataFrame, mde_path, intervalo_m=10.0, fallback_epsg=None):
    _log("=== Iniciando maior_cota_lr() ===")
    lrs_src, raster_crs = align_gdf_to_raster_crs_or_fallback(
        lrs, mde_path, fallback_epsg=fallback_epsg, assume_if_none=True
    )
    with rasterio.open(mde_path) as src:
        mask_img = src.dataset_mask()
        b = src.bounds
        band1_nodata = src.nodata
        transformer_ll = None
        try:
            base_crs = _normalize_crs(raster_crs) or (pyproj.CRS.from_epsg(fallback_epsg) if fallback_epsg else None)
            if base_crs is not None:
                transformer_ll = pyproj.Transformer.from_crs(base_crs, "EPSG:4326", always_xy=True)
        except Exception:
            transformer_ll = None
        resultados = []
        for _, lr in lrs_src.iterrows():
            pts = interpolar_linha_por_metros(lr.geometry, intervalo_m=intervalo_m)
            if not pts:
                resultados.append({"bacia_orig": lr.get("nome_bacia", None), "cota": np.nan, "geometry": None, "status_pl": "sem_dados"})
                continue
            xs = np.asarray([p.x for p in pts], dtype="float64")
            ys = np.asarray([p.y for p in pts], dtype="float64")
            inside = (xs >= b.left) & (xs <= b.right) & (ys >= b.bottom) & (ys <= b.top)
            vals = np.full(xs.shape, np.nan, dtype="float64")
            flags = np.full(xs.shape, "sem_dados", dtype=object)
            zeros_idx = []
            if np.any(inside):
                rr, cc = rowcol(src.transform, xs[inside], ys[inside], op=np.floor)
                rr = np.clip(rr.astype(np.intp), 0, mask_img.shape[0]-1)
                cc = np.clip(cc.astype(np.intp), 0, mask_img.shape[1]-1)
                pix_ok = (mask_img[rr, cc] != 0)
                idx_in = np.where(inside)[0]
                if np.any(pix_ok):
                    v = np.array([vv[0] for vv in src.sample(zip(xs[inside][pix_ok], ys[inside][pix_ok]))], dtype="float64")
                    if band1_nodata is not None:
                        v = np.where(v == band1_nodata, 0.0, v)
                    idx_global_ok = idx_in[pix_ok]
                    vals[idx_global_ok] = v
                    flags[idx_global_ok] = "mdt"
                    zero_mask_ok = (v == 0.0)
                    if np.any(zero_mask_ok):
                        zeros_idx.extend(idx_global_ok[zero_mask_ok].tolist())
                if np.any(~pix_ok):
                    idx_global_nd = idx_in[~pix_ok]
                    vals[idx_global_nd] = 0.0
                    flags[idx_global_nd] = "mdt"
                    zeros_idx.extend(idx_global_nd.tolist())
            fora_idx = np.where(np.isnan(vals))[0].tolist()
            need_fb_idx = sorted(set(zeros_idx) | set(fora_idx))
            if need_fb_idx and (transformer_ll is not None):
                lon, lat = transformer_ll.transform(xs[need_fb_idx], ys[need_fb_idx])
                fb = np.array([buscar_cota_online(lx, ly) for lx, ly in zip(lon, lat)], dtype="float64")
                ok = ~np.isnan(fb)
                if np.any(ok):
                    sel = np.asarray(need_fb_idx)[ok]
                    vals[sel] = fb[ok]
                    flags[sel] = "fallback"
            if np.all(np.isnan(vals)):
                resultados.append({"bacia_orig": lr.get("nome_bacia", None), "cota": np.nan, "geometry": None, "status_pl": "sem_dados"})
                continue
            imax = int(np.nanargmax(vals))
            resultados.append({
                "bacia_orig": lr.get("nome_bacia", None),
                "cota": float(vals[imax]),
                "geometry": pts[imax],
                "status_pl": flags[imax],
            })
        gdf_out = gpd.GeoDataFrame(resultados, crs=raster_crs)
        _log(f"maior_cota_lr() concluído. Linhas processadas: {len(gdf_out)}")
        return gdf_out

# -----------------------------------------
# Montagem da alternativa (extensão + cota + destino)
# -----------------------------------------
def processar_alternativa(df, bacias, lr, pl, sufixo, nome_col_bacia):
    _log(f"=== Processando alternativa {sufixo} ===")
    def obter_ultimo_ponto(geom):
        if geom is None or geom.is_empty:
            return None
        if geom.geom_type == "LineString":
            return Point(geom.coords[-1])
        if geom.geom_type == "MultiLineString":
            ultima = geom.geoms[-1]
            return Point(ultima.coords[-1])
        return None
    ultimos_pontos = [obter_ultimo_ponto(geom) for geom in lr.geometry]
    ultimos_gdf = gpd.GeoDataFrame({"bacia_orig": lr["nome_bacia"]}, geometry=ultimos_pontos, crs=lr.crs)
    ultimos_join = gpd.sjoin(ultimos_gdf, bacias[[nome_col_bacia, "geometry"]], how="left", predicate="within")
    ultimos_join = ultimos_join.rename(columns={nome_col_bacia: f"bacia_destino_{sufixo}"})
    lr_ext = lr.copy()
    lr_ext["extensao_m"] = lr_ext.geometry.length
    temp_df = pl[["bacia_orig", "cota", "status_pl"]].copy()
    temp_df = temp_df.merge(
        lr_ext[["nome_bacia", "extensao_m"]],
        left_on="bacia_orig", right_on="nome_bacia", how="left"
    ).drop(columns="nome_bacia")
    temp_df = temp_df.merge(
        ultimos_join[["bacia_orig", f"bacia_destino_{sufixo}"]],
        on="bacia_orig", how="left"
    )
    temp_df = temp_df.rename(columns={
        "cota": f"cota_pl_{sufixo}",
        "extensao_m": f"extensao_lr_{sufixo}",
        "status_pl": f"status_pl_{sufixo}",
    })
    df_out = df.merge(temp_df, left_on=nome_col_bacia, right_on="bacia_orig", how="left").drop(columns=["bacia_orig"])
    def _consolida(a, b):
        if "mdt" in (a, b): return "mdt"
        if "fallback" in (a, b): return "fallback"
        return "sem_dados"
    df_out[f"status_final_{sufixo}"] = [
        _consolida(a, b)
        for a, b in zip(df_out.get("status_eeb", "sem_dados"), df_out.get(f"status_pl_{sufixo}", "sem_dados"))
    ]
    _log(f"processar_alternativa({sufixo}) concluído.")
    return df_out
# ======================== FIM DO BLOCO ========================


[debug] Raster CRS: EPSG:31982
[debug] Raster bounds (m): (538706.654, 6718197.6829, 544419.654, 6722279.6829)
[debug] EEB CRS lido: EPSG:31982
[debug] EEB bounds (raw): [nan nan nan nan]
[debug] As EEBs parecem estar em METROS (UTM).


## Código Principal

In [7]:
if __name__ == "__main__":
    # ---- entradas ----
    bacias_path = r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\Bacias.gpkg"
    coluna_bacias = "Bacias"

    eeb_path = r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\EEE_Rolante.shp"
    mde_path = os.path.join(r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\Rolante_Mdt.tif")
    caminho_lr = os.path.join(r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\LR_Alt01.gpkg")

    saida = r"C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\SaidaCodigo"
    alt_nome = "Alt1"

    # Corrige CRS das EEB (apenas define, não reprojeta)
    if eeb.crs is None:
        # Detecta pelo range dos números
        minx, miny, maxx, maxy = eeb.total_bounds
        if all([-180 <= v <= 180 for v in [minx, maxx]]) and all([-90 <= v <= 90 for v in [miny, maxy]]):
            eeb = eeb.set_crs("EPSG:4326", allow_override=True)  # ou "EPSG:4674" se for SIRGAS2000 geográfico
            _log("EEB sem CRS: definido como EPSG:4326 (lon/lat).")
        else:
            eeb = eeb.set_crs("EPSG:31982", allow_override=True)  # ajuste se sua UTM for outra zona
            _log("EEB sem CRS: definido como EPSG:31982 (UTM 22S).")
    
    # Corrige CRS das LRs, se necessário
    if lr.crs is None:
        minx, miny, maxx, maxy = lr.total_bounds
        if all([-180 <= v <= 180 for v in [minx, maxx]]) and all([-90 <= v <= 90 for v in [miny, maxy]]):
            lr = lr.set_crs("EPSG:4326", allow_override=True)
            _log("LR sem CRS: definido como EPSG:4326 (lon/lat).")
        else:
            lr = lr.set_crs("EPSG:31982", allow_override=True)
            _log("LR sem CRS: definido como EPSG:31982 (UTM 22S).")

    # ---- sanity checks de arquivo/pasta ----
    for p in [bacias_path, eeb_path, caminho_lr, mde_path]:
        if not os.path.exists(p):
            raise FileNotFoundError(f"Arquivo não encontrado: {p}")
    if not os.path.isdir(saida):
        os.makedirs(saida, exist_ok=True)

    # ---- 0) detectar CRS do MDT ----
    crs_mdt = get_raster_crs(mde_path, fallback_epsg=None)  # não assume 22S; normaliza LOCAL_CS SIRGAS/UTM
    if crs_mdt is None:
        raise RuntimeError("Não foi possível detectar/normalizar o CRS do MDT. Verifique o TIF.")

    # ---- 1) bacias no CRS do MDT ----
    _log(f"Lendo bacias: {bacias_path}")
    bacias = gpd.read_file(bacias_path)
    if coluna_bacias not in bacias.columns:
        raise KeyError(f"Coluna '{coluna_bacias}' não existe no shapefile de bacias.")
    bacias = ensure_to_crs(bacias, crs_mdt, assume_if_none=True)

    # ---- 2) DF base: uma linha por bacia ----
    df = bacias[[coluna_bacias]].drop_duplicates().copy()

    def multipoint_to_point(g):
        if g is None or g.is_empty:
            return None
        # pega o primeiro ponto do MultiPoint
        try:
            first = list(g.geoms)[0]
            return Point(first.x, first.y)
        except Exception:
            return None

    eeb = carregar_shapefile_com_bacia(eeb_path, bacias, coluna_bacias)
    eeb = eeb.set_geometry(eeb.geometry.apply(multipoint_to_point))
    # alinhamento para MDT é feito internamente em adicionar_cota()
    # Em vez de assume_if_none=True, use False aqui:
    eeb = adicionar_cota(eeb, mde_path, fallback_epsg=None)  # <- dentro dela, mude assume_if_none=False
        
  # zeros -> fallback; mantém 0 se fallback falhar

    eeb_por_bacia = (
        eeb[["nome_bacia", "cota", "status_eeb"]]
          .drop_duplicates(subset=["nome_bacia"])
          .rename(columns={"cota": "cota_eeb"})
    )
    df = df.merge(eeb_por_bacia, left_on=coluna_bacias, right_on="nome_bacia", how="left")

    # ---- 4) LRs: associar bacia, alinhar CRS ao MDT e achar maior cota ao longo ----
    lr = carregar_shapefile_com_bacia(caminho_lr, bacias, coluna_bacias)
    lr = ensure_to_crs(lr, crs_mdt, assume_if_none=False)

    pl = maior_cota_lr(lr, mde_path, intervalo_m=1.0, fallback_epsg=None)

    # ---- 5) integrar alternativa (extensão, cota_pl, bacia_destino) ----
    df = processar_alternativa(df, bacias, lr, pl, alt_nome, coluna_bacias)

    # ---- 6) (opcional) trazer status_pl direto pro DF final ----
    df = df.merge(
        pl[["bacia_orig", "status_pl"]].rename(columns={"bacia_orig": coluna_bacias}),
        on=coluna_bacias,
        how="left",
    )

    # ---- 7) salvar ----
    saida_xlsx = os.path.join(saida, f"bacia_destino_{alt_nome}.xlsx")
    df.to_excel(saida_xlsx, index=False)
    _log(f"Planilha gerada com sucesso! -> {saida_xlsx}")


[cotas] CRS do MDT detectado: EPSG:31982
[cotas] Lendo bacias: C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\Bacias.gpkg
[cotas] Lendo shapefile: C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\EEE_Rolante.shp
[cotas] sjoin concluído: 15 feições com bacia.
[cotas] === Iniciando adicionar_cota() ===
[cotas] CRS do MDT detectado: EPSG:31982
[cotas] Pontos que irão ao fallback (fora raster + zeros): 15
[cotas] Transformação raster->WGS84 OK (primeira tentativa).
[cotas] Fallback aplicado em 0 pontos; sem retorno em 15.
[cotas] adicionar_cota() concluído.
[cotas] Lendo shapefile: C:\Users\gabriel.coimbra\Desktop\Meus arquivos\Rolante\LR_Alt01.gpkg
[cotas] sjoin concluído: 15 feições com bacia.
[cotas] === Iniciando maior_cota_lr() ===
[cotas] CRS do MDT detectado: EPSG:31982


KeyboardInterrupt: 